In [ ]:
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, to_hex
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib.ticker import PercentFormatter
from pathlib import Path
from scipy import stats
from sklearn.mixture import GaussianMixture
from typing import Optional, Sequence, Dict, Tuple
import inspect
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scirpy as ir
import seaborn as sns
import warnings


try:
    from kneed import KneeLocator
except Exception:
    KneeLocator = None

Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype']  = 42

warnings.filterwarnings("ignore")


# Fig6b

In [ ]:
GDT = sc.read_h5ad( 'GDT_TCR_filtered.h5ad')


In [ ]:
gdt_color_map = {
    "GD_TN_c5":    "#c8e6c9",
    "GD_TEM_c1":   "#ffa000",
    "GD_TEMRA_c2": "#ff8a65",
    "GD_TEMRA_c3":  "#e53935",
    "GD_TWH_c4":  "#ba68c8",
    "GD_TPro_c6":"#fdd0a2",
    "TRDV3": "#8FAADC",
    "TRDV2": "#A9D18E",
    "TRDV1": "#FFE699",
}


In [ ]:
ax = sc.pl.umap(
    GDT,
    color="IR_VDJ_1_v_call",
    palette=gdt_color_map,
    use_raw=False,
    frameon=False,
    size=25,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig('figures/GDT_delta_chain_umap.pdf', dpi=600, bbox_inches="tight")


# Fig6b

In [ ]:
ax = sc.pl.umap(
    GDT,
    color="anno_GDT",
    palette=gdt_color_map,
    use_raw=False,
    frameon=False,
    size=25,
    legend_loc="right margin",
    show=False,
)

for coll in ax.collections:
    coll.set_rasterized(True)


# Fig6e

In [ ]:
marker_genes_dict = {
    "δchain":["TRDV1","TRDV2","TRDV3","TRGV9"],
    "Naive": ["TCF7","LEF1"],
    "Innate-like":["ZBTB16","KLRB1"],
    "Activation": ["CD69","HLA-DRB1"],
    "Cytolytic": ["GZMK","GZMB","PRF1","GNLY","TNF","IFNG"],
    "Terminal": ["ZEB2","TBX21"],
    "Wound-healing":["AREG","ZNF683"],
    "Type3": ["RORC","CCR6"],
    "Resident": ["ITGAE","ITGA1","CXCR6"],
    "Proliferative":["MKI67","TOP2A"],
}


In [ ]:
base_cmap = mpl.colormaps["coolwarm"]

trunc_cmap = LinearSegmentedColormap.from_list(
    "trunc_coolwarm",
    base_cmap(np.linspace(0.10, 0.90, 256))
)

available_genes = set(GDT.var_names)

filtered_marker_genes_dict = {
    category: [gene for gene in genes if gene in available_genes]
    for category, genes in marker_genes_dict.items()
}

filtered_marker_genes_dict = {
    k: v for k, v in filtered_marker_genes_dict.items() if v
}

if not filtered_marker_genes_dict:
    raise ValueError("没有找到可以用于绘图的基因，请检查 marker_genes_dict 和数据集中的基因名是否匹配。")


In [ ]:
ax=sc.pl.dotplot(
    GDT,
    filtered_marker_genes_dict,
    'anno_GDT',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.show()


In [ ]:
ax=sc.pl.dotplot(
    GDT,
    filtered_marker_genes_dict,
    'IR_VDJ_1_v_call',
    dendrogram=False,
    cmap=trunc_cmap,
    standard_scale="var",
    colorbar_title="column scaled\nexpression",
    use_raw=False,
    return_fig=True,
    show=False
)

ax.show()


# Fig6d

In [ ]:
GDT.obs["Tumor_Blood"] = np.where(
    GDT.obs["Sample_Type"].astype(str) == "T",
    GDT.obs["Disease"].astype(str),
    np.where(
        GDT.obs["Sample_Type"].astype(str) == "B",
        "Blood",
        np.nan
    )
)


In [ ]:
gdt_order = ["TRDV3", "TRDV2", "TRDV1"]
disease_order = ["HB","NB","MB",'PB',"WT","Blood"]
gdt_colors = {
    "TRDV3":  "#8FAADC",
    "TRDV2":  "#A9D18E",
    "TRDV1":  "#FFE699",
}

GDT.obs["IR_VDJ_1_v_call"] = pd.Categorical(GDT.obs["IR_VDJ_1_v_call"], categories=gdt_order, ordered=True)
GDT.obs["Tumor_Blood"] = pd.Categorical(GDT.obs["Tumor_Blood"], categories=disease_order, ordered=True)

prop = (
    pd.crosstab(GDT.obs["Tumor_Blood"], GDT.obs["IR_VDJ_1_v_call"], normalize="index")
      .reindex(index=disease_order)
      .reindex(columns=gdt_order)
      .fillna(0)
)

ax = prop.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    edgecolor="white",
    color=[gdt_colors[g] for g in prop.columns],
    width=0.4
)
ax.set_ylabel("Proportion")
ax.set_xlabel("Disease")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="GDT", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.set_ylim(0, 1)
for label in ax.get_xticklabels():
    label.set_rotation(30)
    label.set_ha("right")
plt.tight_layout()

plt.savefig('figures/GDT_lineage_composition.pdf', dpi=600, bbox_inches="tight")
plt.show()


# Fig6f

In [ ]:
def plot_radars(
    adata,
    class_col="IR_VDJ_1_v_call",
    sample_col="Sample_Type",
    anno_col="annotation_GDT_merged",
    classes=("TRDV1", "TRDV2", "TRDV3"),
    add_overall=True,
    overall_title="All",
    sample_types=("B", "T"),
    sample_colors=None,
    top_n_annos=None,
    title="Cell-type proportions (radar)",
    figsize=None,
    outfile='figures/GDT_state_proportions_radar.pdf',
    r_max=1.0,
    hide_r_tick_labels=True,
    ring_color="#A6AFBC",
    spoke_color="#B8C1CD",
    spine_color="#9099A6",
    ring_lw=1.2,
    spoke_lw=1.0,
    ring_alpha=1.0,
    spoke_alpha=1.0
):

    df = adata.obs[[class_col, sample_col, anno_col]].dropna().copy()

    anno_series_raw = adata.obs.loc[df.index, anno_col]

    for c in (class_col, sample_col):
        df[c] = df[c].astype(str)

    if pd.api.types.is_categorical_dtype(anno_series_raw) and anno_series_raw.cat.ordered:
        anno_order_full = [x for x in anno_series_raw.cat.categories if x in set(anno_series_raw.dropna())]
    else:
        anno_order_full = anno_series_raw.astype(str).value_counts().index.tolist()

    df[anno_col] = anno_series_raw.astype(str)

    if top_n_annos is not None and top_n_annos < len(anno_order_full):
        keep = anno_order_full[:top_n_annos]
        df[anno_col] = np.where(df[anno_col].isin(keep), df[anno_col], "Other")
        anno_order = keep + (["Other"] if (df[anno_col] == "Other").any() else [])
    else:
        anno_order = anno_order_full

    n_anno = len(anno_order)
    if n_anno == 0:
        raise ValueError("没有可用的注释类别。")

    angles = np.linspace(0, 2*np.pi, n_anno, endpoint=False).tolist()
    angles += angles[:1]

    panel_names = list(classes)
    if add_overall:
        panel_names = [overall_title] + panel_names
    n_panels = len(panel_names)

    if figsize is None:
        if n_panels == 4:
            figsize = (16, 7)
        else:
            figsize = (4.6 * n_panels + 4, 5.2)

    plt.rcParams.update({
        "font.size": 11,
        "axes.titlesize": 12,
        "savefig.dpi": 300
    })

    if n_panels == 4:
        fig, axes = plt.subplots(
            2, 2, figsize=figsize,
            subplot_kw={"projection": "polar"},
            sharey=True
        )
        axes = np.array(axes).reshape(-1)
    else:
        fig, axes = plt.subplots(
            1, n_panels, figsize=figsize,
            subplot_kw={"projection": "polar"},
            sharey=True
        )
        if n_panels == 1:
            axes = np.array([axes])

    fig.suptitle(title, y=1.03)
    handle_cache = None

    for ax, name in zip(axes, panel_names):
        if name == overall_title and add_overall:
            sub = df.copy()
        else:
            sub = df[df[class_col] == str(name)].copy()

        ct = pd.crosstab(sub[anno_col], sub[sample_col])
        ct = ct.reindex(index=anno_order, columns=list(sample_types), fill_value=0)

        col_sums = ct.sum(axis=0).replace(0, np.nan)
        props = ct.divide(col_sums, axis=1).fillna(0.0)

        ax.set_facecolor("#FFFFFF")
        ax.spines["polar"].set_color(spine_color)
        ax.spines["polar"].set_linewidth(1.0)
        ax.xaxis.grid(True, color=spoke_color, linewidth=spoke_lw, alpha=spoke_alpha)
        ax.yaxis.grid(True, color=ring_color, linewidth=ring_lw, alpha=ring_alpha)

        ax.set_theta_offset(np.pi / 2)
        ax.set_theta_direction(-1)
        ax.set_xticks(np.linspace(0, 2*np.pi, n_anno, endpoint=False))
        ax.set_xticklabels(anno_order, fontsize=9)

        ax.set_ylim(0, r_max)
        yticks = np.linspace(r_max / 4, r_max, 4)
        ax.set_yticks(yticks)
        if hide_r_tick_labels:
            ax.set_yticklabels([])
        else:
            ax.set_yticklabels([f"{t:.2g}" for t in yticks], fontsize=8, color="#4B5563")

        ax.set_title(name, pad=12, weight="bold")

        handles, labels = [], []
        for st in sample_types:
            vals = props[st].to_list()
            vals += vals[:1]
            color = sample_colors.get(st) if isinstance(sample_colors, dict) else None
            h = ax.plot(angles, vals, linewidth=2.6, color=color, label=st, zorder=3)[0]
            ax.fill(angles, vals, alpha=0.20, color=color, zorder=2)
            handles.append(h)
            labels.append(st)

        if handle_cache is None:
            handle_cache = (handles, labels)

    for ax in axes[n_panels:]:
        ax.axis("off")

    if handle_cache is not None:
        fig.legend(
            *handle_cache,
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            title=sample_col,
            frameon=False
        )

    fig.tight_layout()
    if outfile:
        fig.savefig(outfile, bbox_inches="tight")
    plt.show()


In [ ]:
sample_colors = {"B": "#f2a2af", "T": "#ffc89d"}
plot_radars(
    GDT,
    class_col="IR_VDJ_1_v_call",
    sample_col="Sample_Type",
    anno_col="anno_GDT",
    classes=("TRDV1","TRDV2","TRDV3"),
    sample_types=("B","T"),
    sample_colors=sample_colors,

    add_overall=True,
    figsize=(14,10),
    ring_lw=1.0,
    spoke_lw=1.0,
    ring_alpha=0.5,
    spoke_alpha=0.5,
    r_max=0.8,

)


# Fig6g

In [ ]:
def calc_sample_group_mean_df(
    data,
    group_col="group",
    sample_col="Sample_ID",
    score_col="score",
    group_order=None,
    min_cells_per_sample_group=1,
    sample_meta_cols=None,
    keep_first_meta=True,
):

    if hasattr(data, "obs"):
        obs = data.obs.copy()
    else:
        obs = data.copy()

    required_cols = [group_col, sample_col, score_col]
    missing = [c for c in required_cols if c not in obs.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing required columns: {missing}")

    obs = obs[required_cols + [c for c in (sample_meta_cols or []) if c not in required_cols]].copy()

    obs[group_col] = obs[group_col].astype("string")
    obs[sample_col] = obs[sample_col].astype("string")
    obs = obs.dropna(subset=[group_col, sample_col, score_col])

    df_sample = (
        obs.groupby([sample_col, group_col])[score_col]
        .agg(mean_score="mean", n_cells="size")
        .reset_index()
    )

    df_sample = df_sample.loc[
        df_sample["n_cells"] >= int(min_cells_per_sample_group)
    ].copy()

    if sample_meta_cols is not None and len(sample_meta_cols) > 0:
        meta_cols = [sample_col] + list(sample_meta_cols)
        meta = obs[meta_cols].drop_duplicates().copy()

        dup_counts = meta.groupby(sample_col).size()
        problematic_samples = dup_counts[dup_counts > 1].index.tolist()

        if len(problematic_samples) > 0:
            if keep_first_meta:
                meta = meta.drop_duplicates(subset=[sample_col], keep="first").copy()
            else:
                raise ValueError(
                    "Some samples have multiple metadata rows. "
                    f"Examples: {problematic_samples[:5]}. "
                    "Please check sample_meta_cols or set keep_first_meta=True."
                )

        df_sample = df_sample.merge(meta, on=sample_col, how="left")

    if group_order is None:
        group_order = list(pd.unique(df_sample[group_col]))
    else:
        group_order = list(group_order)

    groups_plot = [g for g in group_order if (df_sample[group_col] == g).any()]
    if len(groups_plot) == 0:
        raise ValueError("No data left after filtering. Check group_order / min_cells_per_sample_group / NA values.")

    df_sample[group_col] = pd.Categorical(
        df_sample[group_col],
        categories=groups_plot,
        ordered=True
    )
    df_sample = df_sample.sort_values([group_col, sample_col]).reset_index(drop=True)

    return df_sample, groups_plot


In [ ]:
def calc_group_paired_stats(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    *,
    group_order: Optional[Sequence[str]] = None,
    paired_col: Optional[str] = None,
    do_unpaired: bool = True,
    do_paired: bool = True,
    test_unpaired: str = "mannwhitney",
    test_paired: str = "wilcoxon",
    alternative: str = "two-sided",
    pair_agg: str = "mean",
):

    required_cols = [value_col, group_col]
    if do_paired:
        if paired_col is None:
            raise ValueError("When do_paired=True, paired_col must be provided.")
        required_cols.append(paired_col)

    missing = [c for c in required_cols if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing required columns: {missing}")

    sub = df[required_cols].dropna(subset=[value_col, group_col]).copy()
    sub[group_col] = sub[group_col].astype("string")
    if paired_col is not None:
        sub[paired_col] = sub[paired_col].astype("string")

    if group_order is None:
        group_order = list(pd.unique(sub[group_col].dropna()))
    else:
        group_order = [g for g in group_order if g in sub[group_col].values]

    if len(group_order) < 2:
        raise ValueError("Need at least two groups for comparison.")

    g1, g2 = str(group_order[0]), str(group_order[1])
    sub2 = sub[sub[group_col].isin([g1, g2])].copy()

    stat_rows = []

    if do_unpaired:
        x1 = sub2.loc[sub2[group_col] == g1, value_col].astype(float).to_numpy()
        x2 = sub2.loc[sub2[group_col] == g2, value_col].astype(float).to_numpy()

        if (len(x1) >= 2) and (len(x2) >= 2):
            if test_unpaired == "ttest":
                res = stats.ttest_ind(x1, x2, equal_var=False, alternative=alternative)
                stat_rows.append(dict(
                    comparison=f"{g1} vs {g2}",
                    mode="unpaired",
                    test="Welch t-test",
                    n_group1=len(x1),
                    n_group2=len(x2),
                    n_pairs=np.nan,
                    statistic=float(res.statistic),
                    pvalue=float(res.pvalue),
                ))
            elif test_unpaired == "mannwhitney":
                res = stats.mannwhitneyu(x1, x2, alternative=alternative)
                stat_rows.append(dict(
                    comparison=f"{g1} vs {g2}",
                    mode="unpaired",
                    test="Mann–Whitney U",
                    n_group1=len(x1),
                    n_group2=len(x2),
                    n_pairs=np.nan,
                    statistic=float(res.statistic),
                    pvalue=float(res.pvalue),
                ))
            else:
                raise ValueError("test_unpaired must be 'ttest' or 'mannwhitney'")
        else:
            stat_rows.append(dict(
                comparison=f"{g1} vs {g2}",
                mode="unpaired",
                test=str(test_unpaired),
                n_group1=len(x1),
                n_group2=len(x2),
                n_pairs=np.nan,
                statistic=np.nan,
                pvalue=np.nan,
            ))

    wide_paired = pd.DataFrame()

    if do_paired:
        if pair_agg == "mean":
            pair_df = (
                sub2.groupby([paired_col, group_col], as_index=False)[value_col]
                .mean()
            )
        elif pair_agg == "median":
            pair_df = (
                sub2.groupby([paired_col, group_col], as_index=False)[value_col]
                .median()
            )
        else:
            raise ValueError("pair_agg must be 'mean' or 'median'")

        wide_paired = pair_df.pivot(
            index=paired_col,
            columns=group_col,
            values=value_col
        )

        keep_cols = [g for g in [g1, g2] if g in wide_paired.columns]
        wide_paired = wide_paired.reindex(columns=keep_cols)

        paired = wide_paired.dropna(subset=[g1, g2]).copy()
        x1p = paired[g1].to_numpy(dtype=float)
        x2p = paired[g2].to_numpy(dtype=float)

        if len(x1p) >= 2:
            if test_paired == "ttest_rel":
                res = stats.ttest_rel(x1p, x2p, alternative=alternative)
                stat_rows.append(dict(
                    comparison=f"{g1} vs {g2}",
                    mode="paired",
                    test="Paired t-test",
                    n_group1=len(x1p),
                    n_group2=len(x2p),
                    n_pairs=len(x1p),
                    statistic=float(res.statistic),
                    pvalue=float(res.pvalue),
                ))
            elif test_paired == "wilcoxon":
                res = stats.wilcoxon(x1p, x2p, alternative=alternative, zero_method="wilcox")
                stat_rows.append(dict(
                    comparison=f"{g1} vs {g2}",
                    mode="paired",
                    test="Wilcoxon signed-rank",
                    n_group1=len(x1p),
                    n_group2=len(x2p),
                    n_pairs=len(x1p),
                    statistic=float(res.statistic),
                    pvalue=float(res.pvalue),
                ))
            else:
                raise ValueError("test_paired must be 'ttest_rel' or 'wilcoxon'")
        else:
            stat_rows.append(dict(
                comparison=f"{g1} vs {g2}",
                mode="paired",
                test=str(test_paired),
                n_group1=len(x1p),
                n_group2=len(x2p),
                n_pairs=len(x1p),
                statistic=np.nan,
                pvalue=np.nan,
            ))

    stat_df = pd.DataFrame(stat_rows)
    return stat_df, wide_paired


In [ ]:
def plot_group_box_dots_paired(
    df: pd.DataFrame,
    value_col: str,
    group_col: str,
    *,
    group_order: Optional[Sequence[str]] = None,
    palette: Optional[Dict[str, str]] = None,
    subgroup_col: Optional[str] = None,
    subgroup_order: Optional[Sequence[str]] = None,
    subgroup_palette: Optional[Dict[str, str]] = None,
    meta_color_map: Optional[Dict[str, str]] = None,
    figsize: Tuple[float, float] = (6, 4),
    jitter: float = 0.08,
    point_size: float = 20,
    min_box_height_ratio: float = 0.03,
    random_state: int = 0,
    ylabel: Optional[str] = None,
    xlabel: Optional[str] = None,
    title: Optional[str] = None,
    show_subgroup_legend: bool = True,
    box_edge_color: str = "#C0C0C0",
    box_line_alpha: float = 0.6,
    box_line_width: float = 1.2,

    connect_paired: bool = False,
    paired_col: Optional[str] = None,
    line_alpha: float = 0.35,
    line_width: float = 1.0,
    line_color: str = "lightgrey",
):

    rng = np.random.default_rng(random_state)

    required_cols = [group_col, value_col]
    if connect_paired:
        if paired_col is None:
            raise ValueError("When connect_paired=True, paired_col must be provided.")
        required_cols.append(paired_col)

    if subgroup_col is not None:
        required_cols.append(subgroup_col)

    missing = [c for c in required_cols if c not in df.columns]
    if len(missing) > 0:
        raise ValueError(f"Missing required columns: {missing}")

    if group_order is None:
        group_order = list(pd.unique(df[group_col].dropna()))
    else:
        group_order = [g for g in group_order if g in df[group_col].values]

    use_cols = [group_col, value_col]
    if subgroup_col is not None:
        use_cols.append(subgroup_col)
    if paired_col is not None:
        use_cols.append(paired_col)

    sub = df[use_cols].dropna(subset=[group_col, value_col]).copy()
    if sub.empty:
        raise ValueError("筛选后数据为空，请检查 value_col / group_col / subgroup_col / paired_col。")

    sub[group_col] = sub[group_col].astype("string")
    if subgroup_col is not None:
        sub[subgroup_col] = sub[subgroup_col].astype("string")
    if paired_col is not None:
        sub[paired_col] = sub[paired_col].astype("string")

    group_order = [g for g in group_order if (sub[group_col] == g).any()]
    if len(group_order) == 0:
        raise ValueError("没有可绘制的 group，请检查 group_order 和数据。")
    if len(group_order) == 1:
        connect_paired = False

    all_vals = sub[value_col].values.astype(float)
    y_min, y_max = float(np.min(all_vals)), float(np.max(all_vals))
    if y_min == y_max:
        y_min -= 0.5
        y_max += 0.5
    y_range = y_max - y_min
    min_box_h = y_range * min_box_height_ratio

    def make_auto_palette(levels, used_colors=None):
        base_tab20 = list(cm.get_cmap("tab20").colors)

        def to_morandi(c, mix=0.85, grey=(0.94, 0.94, 0.94)):
            r, g, b = c[:3]
            return (
                r * mix + grey[0] * (1 - mix),
                g * mix + grey[1] * (1 - mix),
                b * mix + grey[2] * (1 - mix),
            )

        auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if used_colors is not None:
            used_colors = set(c.lower() for c in used_colors)
            auto_colors = [c for c in auto_colors if c.lower() not in used_colors]

        if len(auto_colors) == 0:
            auto_colors = [to_hex(to_morandi(c)) for c in base_tab20]

        if len(auto_colors) < len(levels):
            repeats = int(np.ceil(len(levels) / len(auto_colors)))
            auto_colors = (auto_colors * repeats)[:len(levels)]
        else:
            auto_colors = auto_colors[:len(levels)]

        return dict(zip(levels, auto_colors))

    if palette is None:
        palette = {}
        if meta_color_map is not None:
            for g in group_order:
                if g in meta_color_map:
                    palette[g] = meta_color_map[g]
        for g in group_order:
            if g not in palette:
                palette[g] = box_edge_color

    if subgroup_col is not None:
        if subgroup_order is None:
            sub_levels = list(pd.unique(sub[subgroup_col].dropna()))
        else:
            sub_levels = [s for s in subgroup_order if s in sub[subgroup_col].values]

        final_subgroup_palette = {}

        if meta_color_map is not None:
            for s in sub_levels:
                if s in meta_color_map:
                    final_subgroup_palette[s] = meta_color_map[s]

        if subgroup_palette is not None:
            for s in sub_levels:
                if s in subgroup_palette:
                    final_subgroup_palette[s] = subgroup_palette[s]

        missing_levels = [s for s in sub_levels if s not in final_subgroup_palette]
        if len(missing_levels) > 0:
            auto_pal = make_auto_palette(
                missing_levels,
                used_colors=final_subgroup_palette.values()
            )
            final_subgroup_palette.update(auto_pal)

        subgroup_palette = final_subgroup_palette
    else:
        subgroup_palette = {}
        sub_levels = []

    fig, ax = plt.subplots(figsize=figsize)
    xs = np.arange(len(group_order)) * 0.5
    box_width = 0.30

    whisker_cap_width = box_width / 4

    point_pos = {}

    for x, g in zip(xs, group_order):
        g_mask = sub[group_col] == g
        vals = sub.loc[g_mask, value_col].values.astype(float)
        if len(vals) == 0:
            continue

        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue

        if len(vals) >= 2:
            q1, med, q3 = np.percentile(vals, [25, 50, 75])
        else:
            med = vals[0]
            q1 = med - min_box_h / 2
            q3 = med + min_box_h / 2

        if (q3 - q1) < min_box_h:
            center = (q1 + q3) / 2
            q1 = center - min_box_h / 2
            q3 = center + min_box_h / 2

        w_low = float(np.min(vals))
        w_high = float(np.max(vals))

        box_col = palette.get(g, box_edge_color)

        rect = plt.Rectangle(
            (x - box_width / 2, q1),
            box_width,
            q3 - q1,
            facecolor="none",
            edgecolor=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1,
        )
        ax.add_patch(rect)

        ax.plot(
            [x - box_width / 2, x + box_width / 2],
            [med, med],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=2,
        )

        ax.plot(
            [x, x], [q3, w_high],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x, x], [q1, w_low],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_high, w_high],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1
        )
        ax.plot(
            [x - whisker_cap_width, x + whisker_cap_width],
            [w_low, w_low],
            color=box_col,
            linewidth=box_line_width,
            alpha=box_line_alpha,
            zorder=1
        )

        if subgroup_col is None:
            g_df = sub.loc[g_mask, [value_col] + ([paired_col] if paired_col is not None else [])].copy()
            y = g_df[value_col].values.astype(float)
            jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(y))
            x_coords = np.full_like(y, x, dtype=float) + jitter_x

            ax.scatter(
                x_coords,
                y,
                s=point_size,
                color=box_col,
                alpha=0.85,
                edgecolors="none",
                zorder=3,
            )

            if paired_col is not None:
                for pid, xi, yi in zip(g_df[paired_col].to_numpy(), x_coords, y):
                    point_pos.setdefault((str(pid), str(g)), []).append((float(xi), float(yi)))

        else:
            g_df = sub.loc[g_mask, [value_col, subgroup_col] + ([paired_col] if paired_col is not None else [])].copy()

            for sub_name, sdf in g_df.groupby(subgroup_col):
                v = sdf[value_col].values.astype(float)
                v = v[np.isfinite(v)]
                if len(v) == 0:
                    continue

                jitter_x = rng.normal(loc=0.0, scale=jitter, size=len(v))
                col = subgroup_palette.get(sub_name, "#555555")
                x_coords = np.full_like(v, x, dtype=float) + jitter_x

                ax.scatter(
                    x_coords,
                    v,
                    s=point_size,
                    color=col,
                    alpha=0.9,
                    edgecolors="none",
                    zorder=3,
                    label=sub_name,
                )

                if paired_col is not None:
                    for pid, xi, yi in zip(sdf[paired_col].to_numpy(), x_coords, v):
                        point_pos.setdefault((str(pid), str(g)), []).append((float(xi), float(yi)))

    if connect_paired and (paired_col is not None) and (len(group_order) >= 2):
        g1, g2 = str(group_order[0]), str(group_order[1])

        pair_df = sub[sub[group_col].isin([g1, g2])].copy()
        wide = pair_df.pivot_table(
            index=paired_col,
            columns=group_col,
            values=value_col,
            aggfunc="mean"
        )
        paired_ids = wide.dropna(subset=[g1, g2]).index.astype(str).tolist()

        for pid in paired_ids:
            p1s = point_pos.get((pid, g1), [])
            p2s = point_pos.get((pid, g2), [])
            if (len(p1s) == 0) or (len(p2s) == 0):
                continue

            x1 = np.mean([p[0] for p in p1s])
            y1 = np.mean([p[1] for p in p1s])
            x2 = np.mean([p[0] for p in p2s])
            y2 = np.mean([p[1] for p in p2s])

            ax.plot(
                [x1, x2],
                [y1, y2],
                linewidth=line_width,
                alpha=line_alpha,
                color=line_color,
                zorder=2.5,
            )

    ax.set_xticks(xs)
    ax.set_xticklabels(group_order, rotation=0, ha="center")
    ax.set_ylim(y_min, y_max)
    ax.margins(x=0.05)

    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if title is not None:
        ax.set_title(title)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if subgroup_col is not None and show_subgroup_legend:
        handles, labels = ax.get_legend_handles_labels()
        handle_by_label = dict(zip(labels, handles))

        if subgroup_order is not None:
            legend_labels = [s for s in subgroup_order if s in handle_by_label]
        else:
            legend_labels = [s for s in sub_levels if s in handle_by_label]

        legend_handles = [handle_by_label[l] for l in legend_labels]

        ax.legend(
            legend_handles,
            legend_labels,
            title=subgroup_col,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            frameon=False,
            fontsize="x-small"
        )

    plt.tight_layout()
    return fig, ax


In [ ]:
GDT_tumor=GDT[GDT.obs["Sample_Type"].isin(["T"])].copy()


In [ ]:
vdj = GDT_tumor.obs["IR_VDJ_1_v_call"].astype(str)
GDT_tumor.obs["GD123_T"] = np.where(
    vdj.str.startswith("TRDV2"),
    "GD2T",
    "GD13T"
)
print(GDT_tumor.obs["GD123_T"].value_counts())


In [ ]:
gd_lineage_cytotoxicity_scores, gd_lineage_cytotoxicity_groups = calc_sample_group_mean_df(
    data=GDT_tumor,
    group_col="GD123_T",
    sample_col="Sample_ID",
    score_col="Cytotoxicity",
    group_order=["GD13T", "GD2T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Patient_ID"]
)


In [ ]:
meta_color_map = {
    "GD2T": "#A9D18E",
    "B": "#f2a2af",
    "T": "#ffc89d",
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=gd_lineage_cytotoxicity_scores,
    value_col="mean_score",
    group_col="GD123_T",
    group_order=["GD13T", "GD2T"],
    subgroup_col="Disease",
    meta_color_map=meta_color_map,
    paired_col="Patient_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="Cytotoxicity",
    title="Cytotoxicity"
)
fig.savefig('figures/GDT_lineage_cytotoxicity_box.pdf', dpi=600, bbox_inches="tight")


In [ ]:
gd_lineage_cytotoxicity_stats, gd_lineage_cytotoxicity_paired = calc_group_paired_stats(
    df=gd_lineage_cytotoxicity_scores,
    value_col="mean_score",
    group_col="GD123_T",
    group_order=["GD13T", "GD2T"],
    paired_col="Patient_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

gd_lineage_cytotoxicity_stats


# FigS6d

In [ ]:
gd_lineage_effector_scores, gd_lineage_effector_groups = calc_sample_group_mean_df(
    data=GDT_tumor,
    group_col="GD123_T",
    sample_col="Sample_ID",
    score_col="Activation:Effector function",
    group_order=["GD13T", "GD2T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Patient_ID"]
)


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=gd_lineage_effector_scores,
    value_col="mean_score",
    group_col="GD123_T",
    group_order=["GD13T", "GD2T"],
    subgroup_col="Disease",
    meta_color_map=meta_color_map,
    paired_col="Patient_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="Activation:Effector function",
    title="Activation:Effector function"
)
fig.savefig('figures/GDT_lineage_effector_box.pdf', dpi=600, bbox_inches="tight")


In [ ]:
gd_lineage_effector_stats, gd_lineage_effector_paired = calc_group_paired_stats(
    df=gd_lineage_effector_scores,
    value_col="mean_score",
    group_col="GD123_T",
    group_order=["GD13T", "GD2T"],
    paired_col="Patient_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

gd_lineage_effector_stats


# FigS6e

In [ ]:
VD2_T=GDT[GDT.obs["IR_VDJ_1_v_call"].isin(["TRDV2"])].copy()


In [ ]:
vd2_blood_tumor_effector_scores, vd2_blood_tumor_effector_groups = calc_sample_group_mean_df(
    data=VD2_T,
    group_col="Sample_Type",
    sample_col="Sample_ID",
    score_col="Activation:Effector function",
    group_order=["B", "T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Patient_ID"]
)


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=vd2_blood_tumor_effector_scores,
    value_col="mean_score",
    group_col="Sample_Type",
    group_order=["B", "T"],
    subgroup_col="Disease",
    meta_color_map=meta_color_map,
    paired_col="Patient_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="Activation:Effector function",
    title="Activation:Effector function"
)
fig.savefig('figures/VD2_blood_tumor_effector_box.pdf', dpi=600, bbox_inches="tight")


In [ ]:
vd2_blood_tumor_effector_stats, vd2_blood_tumor_effector_paired = calc_group_paired_stats(
    df=vd2_blood_tumor_effector_scores,
    value_col="mean_score",
    group_col="Sample_Type",
    group_order=["B", "T"],
    paired_col="Patient_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

vd2_blood_tumor_effector_stats


# FigS6f

In [ ]:
vd2_blood_tumor_tcr_scores, vd2_blood_tumor_tcr_groups = calc_sample_group_mean_df(
    data=VD2_T,
    group_col="Sample_Type",
    sample_col="Sample_ID",
    score_col="TCR Signaling",
    group_order=["B", "T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Patient_ID"]
)


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=vd2_blood_tumor_tcr_scores,
    value_col="mean_score",
    group_col="Sample_Type",
    group_order=["B", "T"],
    subgroup_col="Disease",
    meta_color_map=meta_color_map,
    paired_col="Patient_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="TCR Signaling",
    title="TCR Signaling"
)


In [ ]:
vd2_blood_tumor_tcr_stats, vd2_blood_tumor_tcr_paired = calc_group_paired_stats(
    df=vd2_blood_tumor_tcr_scores,
    value_col="mean_score",
    group_col="Sample_Type",
    group_order=["B", "T"],
    paired_col="Patient_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

vd2_blood_tumor_tcr_stats


# FigS6g

In [ ]:
GDT_clone= sc.read_h5ad('GDT_clone.h5ad')

GDT_clone=GDT_clone[GDT_clone.obs["chain_pairing"].isin(["single pair","extra VJ","extra VDJ","two full chains"])]
GDT_clone=GDT_clone.copy()

ir.pp.ir_dist(GDT_clone)
ir.tl.define_clonotypes(GDT_clone, receptor_arms="all", dual_ir="primary_only", within_group="Sample_ID",key_added='clone_Sample')
ir.tl.define_clonotypes(GDT_clone, receptor_arms="all", dual_ir="primary_only", within_group="Patient_ID",key_added='clone_Patient')
GDT_clone.write_h5ad('results/GDT_clonotypes.h5ad', compression='gzip')


In [ ]:
GDT_clone= sc.read_h5ad( 'results/GDT_clonotypes.h5ad')


In [ ]:
def scirpy_expansion_index(
    adata,
    groupby=("patient", "cluster"),
    target_col="clone_id",
    dropna_clonotype=True,
    log_base="e",
    return_details=False,
    neg_epsilon=1e-12,
    min_cells_per_group=None,
    min_clonotypes_per_group=None,
):

    if isinstance(groupby, str):
        group_cols = [groupby]
    else:
        group_cols = list(groupby)

    obs = adata.obs.copy()

    if dropna_clonotype:
        obs = obs[~obs[target_col].isna()].copy()

    cell_counts = (
        obs.groupby(group_cols + [target_col])
           .size()
           .rename("n_cells")
           .reset_index()
    )

    if cell_counts.empty:
        cols = group_cols + [
            "expansion_index",
            "pielou_evenness",
            "shannon",
            "n_clonotypes",
            "n_cells",
        ]
        if return_details:
            return (
                pd.DataFrame(columns=cols),
                pd.DataFrame(columns=group_cols + [target_col, "n_cells", "p"]),
            )
        else:
            return pd.DataFrame(columns=cols)

    cell_counts["group_total"] = cell_counts.groupby(group_cols)["n_cells"].transform("sum")
    cell_counts["p"] = cell_counts["n_cells"] / cell_counts["group_total"]

    if log_base == "e":
        _log = np.log
    elif log_base == 2:
        _log = lambda x: np.log(x) / np.log(2)
    elif log_base == 10:
        _log = lambda x: np.log(x) / np.log(10)
    else:
        raise ValueError("log_base 只能是 'e'、2 或 10")

    rows = []
    for keys, df_g in cell_counts.groupby(group_cols, sort=False):
        keys = (keys,) if not isinstance(keys, tuple) else keys

        p = df_g["p"].to_numpy(dtype=float)
        p = p[p > 0]
        S = int(p.size)

        if S <= 1:
            H = 0.0
            J = np.nan
            EI = np.nan
        else:
            H = float(-(p * _log(p)).sum())
            denom = _log(S)
            J = float(H / denom) if denom > 0 else np.nan
            EI = 1.0 - J

            if EI is not np.nan and EI < 0 and abs(EI) < neg_epsilon:
                EI = 0.0

        n_cells_total = int(df_g["group_total"].iloc[0])
        rows.append((*keys, EI, J, H, S, n_cells_total))

    df_summary = pd.DataFrame(
        rows,
        columns=group_cols
        + ["expansion_index", "pielou_evenness", "shannon", "n_clonotypes", "n_cells"],
    )

    mask = pd.Series(True, index=df_summary.index)

    if min_cells_per_group is not None:
        mask = mask & (df_summary["n_cells"] >= min_cells_per_group)

    if min_clonotypes_per_group is not None:
        mask = mask & (df_summary["n_clonotypes"] >= min_clonotypes_per_group)

    df_summary = df_summary.loc[mask].copy()

    if return_details:
        details = cell_counts[group_cols + [target_col, "n_cells", "p"]].copy()
        if not df_summary.empty:

            valid_groups = df_summary[group_cols].drop_duplicates()
            details = details.merge(valid_groups, on=group_cols, how="inner")
        else:
            details = details.iloc[0:0].copy()
        return df_summary, details

    return df_summary


In [ ]:
GDT_clone.obs["TRDV"] = GDT_clone.obs["IR_VDJ_1_v_call"].copy()


In [ ]:
TRDV = GDT_clone.obs["TRDV"].astype("string")
Sample = GDT_clone.obs["Sample_ID"].astype("string")
GDT_clone.obs["TRDV_Sample"] = (TRDV + "_" + Sample).where(TRDV.notna() & Sample.notna())


In [ ]:
GDT_clone_tumor=GDT_clone[GDT_clone.obs["Sample_Type"].isin(["T"])].copy()
gdt_expansion = scirpy_expansion_index(
    GDT_clone_tumor,
    groupby=("TRDV_Sample"),
    target_col="clone_Sample",
    log_base="e",
    min_cells_per_group=2,
    min_clonotypes_per_group=1,
    return_details=False
)


In [ ]:
extra_cols = [ "Disease","TRDV"]
agg_fun = lambda s: s.mode().iat[0] if not s.mode().empty else (s.dropna().iloc[0] if s.dropna().size else np.nan)

meta = (GDT_clone.obs.groupby("TRDV_Sample")[extra_cols]
          .agg(agg_fun)
          .reset_index())

gdt_expansion_with_meta = gdt_expansion.merge(meta, on="TRDV_Sample", how="left")


In [ ]:
def plot_trdv_boxes_broken_axis(
    df,
    trdv_order=("TRDV1","TRDV2","TRDV3"),
    disease_order=("HB","MB","NB",'PB',"WT"),
    colors_map={"TRDV3": "#8FAADC", "TRDV2": "#A9D18E", "TRDV1": "#FFE699"},
    group_width=0.75, box_width=0.22,
    showfliers=False, whis=1.5,
    edge_alpha=0.55,
    point_size=18, point_jitter=0.06, point_edgecolor="white",
    random_seed=0,

    low_ylim=None, high_ylim=None,
    auto_low_upper_q=95, auto_high_lower_q=99.5, high_pad_ratio=0.05,

    nudge_zeros=1e-6,

    ytick_step=0.01,
    top_yticks=None,
    top_ytick_labels=None,
    figsize=(9, 5.2), save_pdf=None
):
    rng = np.random.default_rng(random_seed)
    df = df.copy()
    df["TRDV"]    = pd.Categorical(df["TRDV"],    categories=list(trdv_order),    ordered=True)
    df["Disease"] = pd.Categorical(df["Disease"], categories=list(disease_order), ordered=True)

    vals_all = df["expansion_index"].astype(float).dropna().values
    if low_ylim is None or high_ylim is None:
        low_upper  = np.nanpercentile(vals_all, auto_low_upper_q)
        high_lower = np.nanpercentile(vals_all, auto_high_lower_q)
        if low_ylim  is None: low_ylim  = (0.0, float(low_upper))
        if high_ylim is None:
            vmax = float(np.nanmax(vals_all))
            high_ylim = (max(float(high_lower), low_ylim[1]*1.001), vmax*(1+high_pad_ratio))
    if not (low_ylim[1] < high_ylim[0]):
        raise ValueError("low_ylim[1] 必须小于 high_ylim[0] 才能形成断轴。")

    pad = 0.001
    low_ylim_display  = (-pad, low_ylim[1])
    high_ylim_display = (high_ylim[0], high_ylim[1])

    fig = plt.figure(figsize=figsize)
    gs = GridSpec(
        2, 2,
        width_ratios=[1.0, 0.16],
        height_ratios=[high_ylim_display[1]-high_ylim_display[0],
                       low_ylim_display[1]-low_ylim_display[0]],
        hspace=0.05, figure=fig
    )
    ax_top = fig.add_subplot(gs[0, 0])
    ax_bot = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_leg = fig.add_subplot(gs[:, 1]); ax_leg.axis("off")

    nD = len(disease_order)
    centers = np.arange(nD) + 1
    k = len(trdv_order)
    offsets = np.linspace(-group_width/2 + box_width/2,
                          group_width/2 - box_width/2, k)

    def draw_layer(ax):
        handles = []
        for i_t, trdv in enumerate(trdv_order):
            pos = centers + offsets[i_t]
            data_per_dis = []
            for dis in disease_order:
                vals = (df.loc[(df["Disease"]==dis) & (df["TRDV"]==trdv),
                               "expansion_index"].dropna().astype(float).values)
                if nudge_zeros and vals.size:
                    vals = vals.copy(); vals[vals == 0.0] = nudge_zeros
                data_per_dis.append(vals)

            edge_rgba = mpl.colors.to_rgba(colors_map.get(trdv, "#BBBBBB"), alpha=edge_alpha)
            bp = ax.boxplot(
                data_per_dis, positions=pos, widths=box_width,
                patch_artist=True, showfliers=showfliers, whis=whis,
                medianprops=dict(linewidth=1.1, color=edge_rgba),
                whiskerprops=dict(linewidth=1.0, color=edge_rgba),
                capprops=dict(linewidth=1.0, color=edge_rgba),
                boxprops=dict(linewidth=1.0, color=edge_rgba),
            )
            for b in bp["boxes"]:
                b.set_facecolor('none'); b.set_edgecolor(edge_rgba)

            for j in range(nD):
                ys = data_per_dis[j]
                if ys.size:
                    xs = np.full_like(ys, pos[j], dtype=float) + rng.normal(0, point_jitter, size=ys.size)
                    ax.scatter(xs, ys, s=point_size,
                               facecolor=colors_map.get(trdv, "#BBBBBB"),
                               edgecolor=point_edgecolor,
                               linewidths=0.4 if point_edgecolor!="none" else 0.0,
                               alpha=1.0, zorder=3)
            handles.append(Rectangle((0,0),1,1, facecolor='none', edgecolor=edge_rgba,
                                     linewidth=1.0, label=trdv))
        return handles

    handles = draw_layer(ax_top)
    draw_layer(ax_bot)

    ax_top.set_ylim(*high_ylim_display)
    ax_bot.set_ylim(*low_ylim_display)

    if top_yticks is not None:
        top_yticks = np.asarray(top_yticks, dtype=float)
        ax_top.set_yticks(top_yticks)
        if top_ytick_labels is not None:
            if len(top_ytick_labels) != len(top_yticks):
                raise ValueError("top_ytick_labels 长度需与 top_yticks 一致。")
            ax_top.set_yticklabels(top_ytick_labels)
        else:
            ax_top.set_yticklabels([f"{t:.2f}" for t in top_yticks])
    else:
        if ytick_step and ytick_step > 0:
            start = np.ceil(high_ylim_display[0] / ytick_step) * ytick_step
            top_ticks = np.arange(start, high_ylim_display[1] + 1e-12, ytick_step)
            ax_top.set_yticks(np.round(top_ticks, 6))

    if ytick_step and ytick_step > 0:
        bot_ticks = np.arange(0.0, low_ylim_display[1] + 1e-12, ytick_step)
        ax_bot.set_yticks(np.round(bot_ticks, 6))
        ax_bot.set_yticklabels([f"{t:.2f}" for t in bot_ticks])

    ax_bot.set_xticks(centers); ax_bot.set_xticklabels(disease_order)
    ax_bot.set_xlim(centers[0]-0.9, centers[-1]+0.9)
    ax_top.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    ax_top.spines['bottom'].set_visible(False)

    for ax in (ax_top, ax_bot):
        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    ax_top.spines['left'].set_visible(True)
    ax_bot.spines['left'].set_visible(True)
    ax_bot.spines['bottom'].set_visible(True)
    ax_bot.set_ylabel("expansion_index")

    ax_leg.legend(handles=handles, title="TRDV", frameon=False, ncol=1,
                  loc='upper left', bbox_to_anchor=(0.0, 1.0), borderaxespad=0.0)

    fig.tight_layout()
    if save_pdf:
        fig.savefig(save_pdf, format="pdf", bbox_inches="tight")
        print(f"Saved PDF to: {save_pdf}")
    return fig, (ax_top, ax_bot, ax_leg)


In [ ]:
plot_trdv_boxes_broken_axis(
    gdt_expansion_with_meta,
    trdv_order=("TRDV1","TRDV2","TRDV3"),
    disease_order=("HB","MB","NB",'PB',"WT"),

    low_ylim=(0.0, 0.05), high_ylim=(0.14, 0.15),ytick_step=0.01,top_yticks=[0.14, 0.15],
    figsize=(7, 2),
   save_pdf='figures/GDT_expansion_index.pdf'
)


# Fig6h

In [ ]:
T_all= sc.read_h5ad( 'T.h5ad')


In [ ]:
T_tumor_reference=T_all[T_all.obs["Sample_Type"].isin(["T"])].copy()
T_tumor_reference=T_tumor_reference[T_tumor_reference.obs["Origin"].isin(["Dong_lab"])].copy()


In [ ]:
GDT_comparison = sc.read_h5ad( 'GDT_TCR_filtered.h5ad')
vdj = GDT_comparison.obs["IR_VDJ_1_v_call"].astype(str)

GDT_comparison.obs["GD123_T"] = np.where(
    vdj.str.startswith("TRDV2"),
    "GD2T",
    "GD13T"
)

print(GDT_comparison.obs["GD123_T"].value_counts())


In [ ]:
tcr_comparison=sc.read_h5ad('tcr.h5ad')
ir.pp.merge_with_ir(T_tumor_reference,tcr_comparison)
ir.tl.chain_qc(T_tumor_reference)


In [ ]:
vj  = T_tumor_reference.obs["IR_VJ_1_v_call"].astype(str)

T_tumor_reference.obs["AB_GD_T"] = np.where(
    vj.str.startswith("TRAV") & (vj != "TRAV1-2"),
    "ABT",
    "other"
)


In [ ]:
vdj = GDT_comparison.obs["IR_VDJ_1_v_call"].astype(str)

GDT_comparison.obs["AB_GD_T"] = np.where(
    vdj.str.startswith("TRDV2"),
    "GD2T",
    "other"
)


In [ ]:
common_cells = T_tumor_reference.obs_names.intersection(GDT_comparison.obs_names)

print(f"共有 {len(common_cells)} 个共同细胞")

if "AB_GD_T" not in T_tumor_reference.obs.columns:
    T_tumor_reference.obs["AB_GD_T"] = "other"

T_tumor_reference.obs.loc[common_cells, "AB_GD_T"] = GDT_comparison.obs.loc[common_cells, "AB_GD_T"].values

print(T_tumor_reference.obs["AB_GD_T"].value_counts())


In [ ]:
CD8_VD2_tumor = T_tumor_reference[~T_tumor_reference.obs["AB_GD_T"].isin(["other"])].copy()


In [ ]:
CD8_VD2_tumor = CD8_VD2_tumor[
    (CD8_VD2_tumor.obs["AB_GD_T"] == "GD2T") |
    (CD8_VD2_tumor.obs["T_sub_level_1"] == "CD8_T")
].copy()


In [ ]:
CD8_VD2_tumor.obs["Sample_AB_GD_T"] = (
    CD8_VD2_tumor.obs["Sample_ID"].astype(str) + "_" +
    CD8_VD2_tumor.obs["AB_GD_T"].astype(str)
)


In [ ]:
cd8_vd2_cytotoxicity_scores, cd8_vd2_cytotoxicity_groups = calc_sample_group_mean_df(
    data=CD8_VD2_tumor,
    group_col="AB_GD_T",
    sample_col="Sample_AB_GD_T",
    score_col="Cytotoxicity",
    group_order=["ABT", "GD2T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Sample_ID"]
)


In [ ]:
abgd_color_map = {
    "ABT": "#9edae5",
    "GD2T": "#A9D18E",
    "NB": "#e6550d",
    "HB":  "#fd8d3c",
    "WT":  "#ffed6f",
    'PB': "#ffffb3" ,
    "MB":  "#e7969c",
    'pLGG':  "#d6616b",
    'pHGG':  "#ad494a",
    "BRCA":"#3182bd",
    'CRC':"#6baed6",
    'PC':"#8ca252",
    "LC":"#b5cf6b",
    "HCC":"#756BB1",
    'RC':"#bebada",
    'aHCC':"#a55194",
}


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=cd8_vd2_cytotoxicity_scores,
    value_col="mean_score",
    group_col="AB_GD_T",
    group_order=["ABT", "GD2T"],
    subgroup_col="Disease",
    meta_color_map=abgd_color_map,
    paired_col="Sample_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="Cytotoxicity",
    title="Cytotoxicity"
)


In [ ]:
cd8_vd2_cytotoxicity_stats, cd8_vd2_cytotoxicity_paired = calc_group_paired_stats(
    df=cd8_vd2_cytotoxicity_scores,
    value_col="mean_score",
    group_col="AB_GD_T",
    group_order=["ABT", "GD2T"],
    paired_col="Sample_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

cd8_vd2_cytotoxicity_stats


# FigS6h

In [ ]:
cd8_vd2_effector_scores, cd8_vd2_effector_groups = calc_sample_group_mean_df(
    data=CD8_VD2_tumor,
    group_col="AB_GD_T",
    sample_col="Sample_AB_GD_T",
    score_col="Activation:Effector function",
    group_order=["ABT", "GD2T"],
    min_cells_per_sample_group=3,
    sample_meta_cols=["Disease","Sample_ID"]
)


In [ ]:
fig, ax = plot_group_box_dots_paired(
    df=cd8_vd2_effector_scores,
    value_col="mean_score",
    group_col="AB_GD_T",
    group_order=["ABT", "GD2T"],
    subgroup_col="Disease",
    meta_color_map=abgd_color_map,
    paired_col="Sample_ID",
    connect_paired=True,
    figsize=(3.5, 4),
    ylabel="Activation:Effector function",
    title="Activation:Effector function"
)


In [ ]:
cd8_vd2_effector_stats, cd8_vd2_effector_paired = calc_group_paired_stats(
    df=cd8_vd2_effector_scores,
    value_col="mean_score",
    group_col="AB_GD_T",
    group_order=["ABT", "GD2T"],
    paired_col="Sample_ID",
    do_unpaired=True,
    do_paired=True,
    test_unpaired="mannwhitney",
    test_paired="wilcoxon",
    alternative="two-sided",
    pair_agg="mean",
)

cd8_vd2_effector_stats


# FigS6a

In [ ]:
T_qc= sc.read_h5ad('T.h5ad')
T_qc=T_qc[T_qc.obs["Origin"].isin(["Dong_lab"])].copy()


In [ ]:
abtcr=sc.read_h5ad( 'donglab_abtcr_all.h5ad')
gdtcr=sc.read_h5ad( 'donglab_gdtcr_all.h5ad')


In [ ]:
ab_common_barcodes = T_qc.obs.index.intersection(abtcr.obs.index)
gd_common_barcodes = T_qc.obs.index.intersection(gdtcr.obs.index)
print("ab_common barcodes:", len(ab_common_barcodes))
print("gd_common barcodes:", len(gd_common_barcodes))

abtcr_matched = abtcr[ab_common_barcodes].copy()
gdtcr_matched = gdtcr[gd_common_barcodes].copy()


In [ ]:
ir.tl.chain_qc(abtcr_matched)
ir.tl.chain_qc(gdtcr_matched)


In [ ]:
ab_map = {
    "orphan VJ": "a",
    "orphan VDJ": "b",
    "single pair": "a+b",
    "extra VJ": "a+b",
    "extra VDJ": "a+b",
    "two full chains": "a+b"
}
abtcr_matched.obs["chain_ab"] = abtcr_matched.obs["chain_pairing"].map(ab_map)


In [ ]:
gd_map = {
    "orphan VJ": "g",
    "orphan VDJ": "d",
    "single pair": "g+d",
    "extra VJ": "g+d",
    "extra VDJ": "g+d",
    "two full chains": "g+d"
}
gdtcr_matched.obs["chain_gd"] = gdtcr_matched.obs["chain_pairing"].map(gd_map)


In [ ]:
T_chain_pair = pd.DataFrame(index=T_qc.obs.index)

T_chain_pair["chain_ab"] = abtcr_matched.obs["chain_ab"].reindex(T_chain_pair.index)

T_chain_pair["chain_gd"] = gdtcr_matched.obs["chain_gd"].reindex(T_chain_pair.index)

T_chain_pair["chain_pair"] = ""

chain_ab = T_chain_pair["chain_ab"].fillna("")
chain_gd = T_chain_pair["chain_gd"].fillna("")

T_chain_pair["chain_pair"] = chain_ab + "+" + chain_gd

T_chain_pair["chain_pair"] = (
    T_chain_pair["chain_pair"]
    .str.strip("+")
    .replace("", "none")
)


In [ ]:
chain_pair_summary = (
    T_chain_pair["chain_pair"]
    .value_counts(dropna=False)
    .rename_axis("chain_pair")
    .reset_index(name="count")
)

plt.figure(figsize=(5, 5))

plt.bar(
    chain_pair_summary["chain_pair"],
    chain_pair_summary["count"],
    color="#0277bd"
)

plt.xlabel("chain_pair")
plt.ylabel("Cell count")
plt.title("Distribution of chain_pair in T cells")
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig('figures/TCR_chain_combinations.pdf', bbox_inches="tight")
plt.show()


# FigS6b

In [ ]:
T_atlas= sc.read_h5ad( 'T.h5ad')


In [ ]:
tcr_atlas=sc.read_h5ad('tcr.h5ad')
ir.pp.merge_with_ir(T_atlas,tcr_atlas)
ir.tl.chain_qc(T_atlas)


In [ ]:
receptor_map = {
    "TRDV1": "TRDV1/3",
    "TRDV2": "TRDV2",
    "TRDV3": "TRDV1/3",
}
T_atlas.obs['Dchain_capture'] = T_atlas.obs['IR_VDJ_1_v_call'].map(receptor_map).fillna("undetected")

pd.set_option('display.max_rows', None)

print(T_atlas.obs['Dchain_capture'].value_counts())


In [ ]:
viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(T_atlas.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
umap_df['Dchain_capture'] = T_atlas.obs['Dchain_capture'].values

subset_all = umap_df

subset_gd2t = umap_df[umap_df['Dchain_capture'] == 'TRDV2']

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.1
)

plt.figure(figsize=(7, 6))

sns.kdeplot(
    data=subset_all,
    x='UMAP1',
    y='UMAP2',
    **kws_background
)

sns.kdeplot(
    data=subset_gd2t,
    x='UMAP1',
    y='UMAP2',
    **kws_foreground
)

x_min, x_max = subset_all['UMAP1'].min(), subset_all['UMAP1'].max()
y_min, y_max = subset_all['UMAP2'].min(), subset_all['UMAP2'].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

plt.xlim(x_min - x_margin, x_max + x_margin)
plt.ylim(y_min - y_margin, y_max + y_margin)

plt.tight_layout()

plt.gca().set_axis_off()
plt.savefig('figures/delta2_capture_density.tiff', format="tiff", dpi=300, bbox_inches="tight",transparent=True)
plt.show()


In [ ]:
viridis = mpl.cm.get_cmap("viridis")

umap_df = pd.DataFrame(T_atlas.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
umap_df['Dchain_capture'] = T_atlas.obs['Dchain_capture'].values

subset_all = umap_df

subset_gd13t = umap_df[umap_df['Dchain_capture'] == 'TRDV1/3']

kws_background = dict(
    fill=True,
    cmap=mpl.colors.ListedColormap([viridis(0.0)]),
    bw_adjust=0.7,
    levels=50,
    thresh=0.01
)

kws_foreground = dict(
    fill=True,
    cmap="viridis",
    bw_adjust=0.5,
    levels=50,
    thresh=0.1
)

plt.figure(figsize=(7, 6))

sns.kdeplot(
    data=subset_all,
    x='UMAP1',
    y='UMAP2',
    **kws_background
)

sns.kdeplot(
    data=subset_gd13t,
    x='UMAP1',
    y='UMAP2',
    **kws_foreground
)

x_min, x_max = subset_all['UMAP1'].min(), subset_all['UMAP1'].max()
y_min, y_max = subset_all['UMAP2'].min(), subset_all['UMAP2'].max()
x_margin = (x_max - x_min) * 0.05
y_margin = (y_max - y_min) * 0.05

plt.xlim(x_min - x_margin, x_max + x_margin)
plt.ylim(y_min - y_margin, y_max + y_margin)

plt.tight_layout()

plt.gca().set_axis_off()
plt.savefig('figures/delta13_capture_density.tiff', format="tiff", dpi=300, bbox_inches="tight",transparent=True)
plt.show()


# FigS6c


In [ ]:
T_expression = T_atlas[T_atlas.obs["Sample_Type"].isin(["T"])].copy()
T_expression=T_expression[T_expression.obs["Sequence"].isin(["10X5"])].copy()


In [ ]:
def _gmm_fit_with_optional_weights(gm, x, w):
    sig = inspect.signature(gm.fit)
    if "sample_weight" in sig.parameters:
        return gm.fit(x, sample_weight=w)

    w = np.asarray(w, dtype=float)
    total = int(w.sum())
    if total <= 0:
        return gm.fit(x)

    max_points = 50000
    scale = min(1.0, max_points / total)
    reps = np.maximum(1, np.round(w * scale).astype(int))
    x_rep = np.repeat(x, reps, axis=0)
    return gm.fit(x_rep)

def cutoff_gmm_intersection(score, bins=256, reg_covar=1e-6, max_iter=200, tol=1e-3):
    score = np.asarray(score)
    score = score[np.isfinite(score)]
    if score.size == 0:
        return np.nan
    if np.all(score == score[0]):
        return float(score[0])

    hist, edges = np.histogram(score, bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    x = centers.reshape(-1, 1)
    w = hist.astype(float)

    q30, q70 = np.percentile(score, [30, 70])
    gm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        reg_covar=reg_covar,
        max_iter=max_iter,
        tol=tol,
        init_params="kmeans",
        random_state=0,
        means_init=np.array([[q30], [q70]]),
    )
    gm = _gmm_fit_with_optional_weights(gm, x, w)

    wt = gm.weights_
    mu = gm.means_.ravel()
    var = np.array([np.diag(cov)[0] for cov in gm.covariances_])

    a = 1/(2*var[0]) - 1/(2*var[1])
    b = mu[1]/var[1] - mu[0]/var[0]
    c = (mu[0]**2)/(2*var[0]) - (mu[1]**2)/(2*var[1]) + np.log((wt[1]*np.sqrt(var[0]))/(wt[0]*np.sqrt(var[1])))

    disc = b*b - 4*a*c
    if disc < 0 or (a == 0 and b == 0):
        return float(np.mean(mu))

    if a != 0:
        r1 = (-b + np.sqrt(disc)) / (2*a)
        r2 = (-b - np.sqrt(disc)) / (2*a)
    else:
        r1 = r2 = -c / b

    lo, hi = float(np.min(mu)), float(np.max(mu))
    cand = [r for r in (r1, r2) if lo <= r <= hi]
    return float(cand[0] if cand else np.mean(mu))

def cutoff_knee(score):
    score = np.asarray(score)
    score = score[np.isfinite(score)]
    if score.size == 0:
        return np.nan
    s = np.sort(score)
    try:
        x = np.arange(len(s))
        knee = KneeLocator(x, s, curve="convex", direction="increasing")
        return float(s[knee.knee]) if knee.knee is not None else float(np.percentile(s, 95))
    except Exception:
        d1 = np.gradient(s)
        d2 = np.gradient(d1)
        idx = int(np.argmax(d2))
        return float(s[idx])

def cutoff_otsu(
    score,
    bins=256,
    only_positive=False,
    min_positive=0.0,
    min_pos_n=30,
    fallback="percentile:90",
):
    score = np.asarray(score)
    score = score[np.isfinite(score)]
    if score.size == 0:
        return np.nan
    if np.all(score == score[0]):
        return float(score[0])

    if only_positive:
        pos = score[score > min_positive]
        if pos.size < min_pos_n:

            if fallback == "nan":
                return np.nan
            if fallback.startswith("percentile:"):
                q = float(fallback.split(":", 1)[1])
                return float(np.percentile(pos, q)) if pos.size else np.nan
            if fallback == "min_positive":
                return float(min_positive)
            raise ValueError("fallback must be one of: nan | min_positive | percentile:XX")
        work = pos
    else:
        work = score

    hist, edges = np.histogram(work, bins=bins)
    if hist.sum() == 0:
        return np.nan

    p = hist / (hist.sum() + 1e-12)
    omega = np.cumsum(p)
    centers = edges[:-1] + np.diff(edges) / 2
    mu = np.cumsum(p * centers)
    mu_t = mu[-1]

    sigma_b2 = (mu_t * omega - mu) ** 2 / (omega * (1 - omega) + 1e-12)
    k = int(np.nanargmax(sigma_b2))

    thr = float(centers[k])

    if only_positive and thr <= min_positive:
        if fallback.startswith("percentile:"):
            q = float(fallback.split(":", 1)[1])
            return float(np.percentile(work, q))
        return float(min_positive)

    return thr

def pick_cutoff(score, method="gmm"):
    if method == "gmm":
        return cutoff_gmm_intersection(score)
    if method == "knee":
        return cutoff_knee(score)
    if method == "otsu":
        return cutoff_otsu(score)
    if isinstance(method, str) and method.endswith("%"):
        q = float(method.strip("%"))
        return float(np.percentile(np.asarray(score)[np.isfinite(score)], q))
    raise ValueError("method must be one of: gmm | knee | otsu | '90%'/'95%' ...")


In [ ]:
def add_expr_and_get_cutoffs(
    adata,
    genes,
    method="otsu",
    make_posneg=False,
    posneg_suffix="_pos",
    batch_key=None,
    min_cells_per_batch=30,
    fallback="global",
    return_global_if_batch=True,
    layer=None,
    use_raw=False,
):

    genes = list(genes)

    if use_raw:
        if adata.raw is None:
            raise ValueError("use_raw=True but adata.raw is None")
        var_names = adata.raw.var_names
    else:
        var_names = adata.var_names

    present = [g for g in genes if g in var_names]
    missing = [g for g in genes if g not in var_names]
    if missing:
        print(f"[add_expr_and_get_cutoffs] Missing genes skipped: {missing}")
    if len(present) == 0:
        raise ValueError("None of the input genes are in the selected var_names")

    if use_raw:
        X = adata.raw[:, present].X
    else:
        if layer is None:
            X = adata[:, present].X
        else:
            if layer not in adata.layers:
                raise ValueError(f"layer='{layer}' not found in adata.layers")

            gene_idx = np.array([adata.var_names.get_loc(g) for g in present], dtype=int)
            X = adata.layers[layer][:, gene_idx]

    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

    if batch_key is not None:
        if batch_key not in adata.obs.columns:
            raise ValueError(f"batch_key='{batch_key}' not found in adata.obs")
        batch = np.asarray(adata.obs[batch_key])
    else:
        batch = None

    global_cutoffs = {}
    by_batch_cutoffs = {}

    def _fallback_cut(score_all, score_sub):
        if fallback == "nan":
            return np.nan
        if fallback == "global":
            return pick_cutoff(score_all, method=method)
        if isinstance(fallback, str) and fallback.startswith("percentile:"):
            q = float(fallback.split(":", 1)[1])
            s = np.asarray(score_sub)
            s = s[np.isfinite(s)]
            return float(np.percentile(s, q)) if s.size else np.nan
        raise ValueError("fallback must be one of: global | nan | percentile:95 (etc)")

    for i, g in enumerate(present):
        score_all = X[:, i]
        adata.obs[f"{g}_exp"] = score_all

        g_cut_global = pick_cutoff(score_all, method=method)
        global_cutoffs[g] = float(g_cut_global) if np.isfinite(g_cut_global) else np.nan

        if batch_key is None:
            if make_posneg:
                adata.obs[f"{g}{posneg_suffix}"] = np.where(
                    score_all >= global_cutoffs[g], f"{g}_pos", f"{g}_neg"
                )
            continue

        by_batch_cutoffs[g] = {}
        for b in np.unique(batch):
            mask = (batch == b)
            score_sub = score_all[mask]
            score_sub_f = np.asarray(score_sub)[np.isfinite(score_sub)]
            if score_sub_f.size < min_cells_per_batch:
                cut = _fallback_cut(score_all, score_sub)
            else:
                cut = pick_cutoff(score_sub, method=method)
            by_batch_cutoffs[g][b] = float(cut) if np.isfinite(cut) else np.nan

        if make_posneg:

            cut_dict = by_batch_cutoffs[g]
            thr = np.array([cut_dict.get(bb, np.nan) for bb in batch], dtype=float)

            adata.obs[f"{g}{posneg_suffix}"] = np.where(
                score_all >= thr, f"{g}_pos", f"{g}_neg"
            )

    if batch_key is None:
        return global_cutoffs

    return {"global": global_cutoffs, "by_batch": by_batch_cutoffs} if return_global_if_batch else by_batch_cutoffs


In [ ]:
cuts = add_expr_and_get_cutoffs(
    T_expression,
    genes=["TRDV1","TRDV2","TRDV3"],
    method="otsu",
    batch_key=None,
    make_posneg=True,
    layer="raw.X.log_norm",
    use_raw=False
)


In [ ]:
T_expression.obs["TCRD"] = (
    T_expression.obs["TRDV1_pos"].astype(str) + "_" +
    T_expression.obs["TRDV2_pos"].astype(str) + "_" +
    T_expression.obs["TRDV3_pos"].astype(str)
)


In [ ]:
T_expression.obs["TCRD_class"] = "Other"

mapping = {
    "TRDV1_pos_TRDV2_neg_TRDV3_neg": "TRDV1",
    "TRDV1_neg_TRDV2_pos_TRDV3_neg": "TRDV2",
    "TRDV1_neg_TRDV2_neg_TRDV3_pos": "TRDV3",
}

T_expression.obs["TCRD_class"] = T_expression.obs["TCRD"].map(mapping).fillna("Other")

T_expression.obs["TCRD_class"] = T_expression.obs["TCRD_class"].astype("category")
T_expression.obs["TCRD_class"] = T_expression.obs["TCRD_class"].cat.set_categories(
    ["TRDV1", "TRDV2", "TRDV3", "Other"],
    ordered=True
)

print(T_expression.obs["TCRD_class"].value_counts())


In [ ]:
gdt_order = ["Other","TRDV3", "TRDV2", "TRDV1"]
disease_order = ['pHGG','pLGG',"MB","NB","HB","WT",'PB','aHCC',"LC","BRCA","HCC",'RC','PC',"CRC"]
gdt_colors = {
    "Other": "#DBDBDB",
    "TRDV3":  "#8FAADC",
    "TRDV2":  "#A9D18E",
    "TRDV1":  "#FFE699",
}

T_expression.obs["TCRD_class"] = pd.Categorical(T_expression.obs["TCRD_class"], categories=gdt_order, ordered=True)
T_expression.obs["Disease"] = pd.Categorical(T_expression.obs["Disease"], categories=disease_order, ordered=True)

prop = (
    pd.crosstab(T_expression.obs["Disease"], T_expression.obs["TCRD_class"], normalize="index")
      .reindex(index=disease_order)
      .reindex(columns=gdt_order)
      .fillna(0)
)

ax = prop.plot(
    kind="bar",
    stacked=True,
    figsize=(8, 5),
    edgecolor="white",
    color=[gdt_colors[g] for g in prop.columns],
    width=0.4
)
ax.set_ylabel("Proportion")
ax.set_xlabel("Disease")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(title="GDT", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
ax.set_ylim(0, 1)
for label in ax.get_xticklabels():
    label.set_rotation(30)
    label.set_ha("right")
plt.tight_layout()

plt.show()
